# Notebook 04 — Dataset Creation
### WID2003 Cognitive Science | FSKTM, Universiti Malaya

---

## Overview

Before training a classifier, features and labels must be merged into a single, clean, analysis-ready table. This notebook reshapes the long-format per-task features into a wide participant-level dataset, handles missing data, caps outliers, and applies standardization.

**Inputs**
| File | Description |
|---|---|
| `data/processed/features_per_task.parquet` | Eye-tracking features — one row per participant × task |
| `data/processed/labels.csv` | Performance label per participant |

**Output**
| File | Description |
|---|---|
| `data/processed/dataset_final.parquet` | One row per participant, ~450+ feature columns, ready for modelling |
| `data/processed/scaler.pkl` | Fitted `StandardScaler` (trained on training rows only) |

---

## Learning Objectives

By the end of this notebook, you should be able to:

1. Reshape data from **long format** (participant × task rows) to **wide format** (one row per participant) using `pivot_table`
2. Explain why a `StandardScaler` must be **fit on the training set only** and only applied (transformed) to the test set — and describe what goes wrong if you fit on both
3. Apply **median imputation** for missing values and justify when this is appropriate vs. when it is not
4. Describe the **1.5×IQR rule** for outlier capping and explain why capping is preferred over deletion in small datasets
5. Implement a **stratified train/test split** and explain why stratification matters for imbalanced class labels

---

## Background

### Long-to-Wide Pivoting

After feature extraction, data is stored in **long format**: each row is one (participant, task) pair, and feature columns contain values specific to that task. Machine learning models expect **wide format**: one row per participant with all features as separate columns.

The pivot operation creates column names like:
```
Crown_correct_Total_duration_of_fixations
Crown_correct_aoi_dwell_ratio
Toothbrush_correct_Time_to_first_fixation
...
```

This results in approximately **13 tasks × ~35 features = ~450 per-task feature columns**, plus a handful of cross-task aggregate features.

### Missing Data Strategies

Not every participant has data for every AOI on every task (e.g., if they never looked at the answer region). This produces `NaN` values in the wide table.

| Strategy | When to use |
|---|---|
| Median imputation | < 30% missing in a column; distribution is skewed |
| Mean imputation | < 30% missing; distribution is symmetric |
| Drop the column | ≥ 30% missing; too little information to be useful |
| Drop the participant | Participant has nearly all features missing |

We use **median imputation** because many eye-tracking metrics are right-skewed (a few very long fixation durations inflate the mean).

### Outlier Capping (1.5×IQR)

Extreme values can dominate a classifier. The **IQR rule** identifies the interquartile range:

```
lower fence = Q1 − 1.5 × IQR
upper fence = Q3 + 1.5 × IQR
```

Values outside these fences are **clipped** (pulled to the fence value), not deleted. This is preferred in small samples because deletion wastes data.

### Feature Scaling

`StandardScaler` transforms each feature to zero mean and unit variance:

```
z = (x − mean) / std
```

**Critical rule:** fit the scaler on **training rows only**, then apply (`transform`) to both train and test rows. Fitting on all rows leaks test statistics into training — a form of data leakage that artificially inflates cross-validation scores.

### Stratified Train/Test Split

A random split may, by chance, put all High performers in training and all Low performers in test (or vice versa). A **stratified split** ensures the class ratio is preserved in both partitions. With very small N (< 10), there may not be enough data for a hold-out test set — in that case, all participants go into cross-validation and the `split` column is set to `train` for all rows.

---

## Discussion Questions

1. You pivot features from long to wide format, creating ~450 columns from 13 tasks. If a student was absent for one task, approximately how many columns will be `NaN`? How does this affect the imputation strategy?
2. A classmate proposes fitting `StandardScaler` on the full dataset to "get better scaling statistics." Explain specifically what goes wrong with this approach and give a concrete example of how it leads to overly optimistic evaluation results.
3. The 1.5×IQR rule flags outliers based on the distribution of the training data. If a test participant has a genuinely extreme fixation duration (e.g., they stared at the wrong region for 30 seconds), should this be capped? What are the arguments for and against?
4. Cross-task aggregate features (mean fixation duration across all 13 tasks) are computed before the train/test split. Does this introduce data leakage? Justify your answer.
5. With 130 participants, a 20% test set contains ~26 people. How does this compare to a small-sample study (20–30 participants total)? What evaluation choices become available with a larger N that weren't with a smaller one?

In [1]:
# from google.colab import drive
# drive.mount("/content/drive")

# %cd /content/drive/MyDrive/WID2003

# !python -m pip install -q -r requirements.txt

# # Important: notebook imports assume the working directory is notebooks/
# %cd /content/drive/MyDrive/WID2003/notebooks

In [2]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from src.config import (
    FEATURES_PER_TASK, LABELS_CSV, DATASET_FINAL, SCALER_PKL,
    PERFORMANCE_LABEL_COL, RANDOM_STATE, TEST_SIZE
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [3]:
features_long = pd.read_parquet(FEATURES_PER_TASK)
labels        = pd.read_csv(LABELS_CSV)

print(f"Features (long): {features_long.shape}")
print(f"Labels:          {labels.shape}")
print(f"Participants in features: {features_long['participant_id'].nunique()}")
print(f"Participants in labels:   {labels['participant_id'].nunique()}")

Features (long): (1690, 37)
Labels:          (130, 19)
Participants in features: 130
Participants in labels:   130


## 2. Pivot features from long to wide format

In [4]:
# Identify feature columns (everything except participant_id and task)
feature_cols = [c for c in features_long.columns if c not in ['participant_id', 'task']]

# Pivot: create columns like "findDice_correct_Total_duration_of_fixations"
features_wide = features_long.pivot_table(
    index='participant_id',
    columns='task',
    values=feature_cols,
    aggfunc='mean'  # each participant should have one row per task; mean handles any duplicates
)

# Flatten MultiIndex columns: (feature, task) → "task_feature"
features_wide.columns = [f"{task}_{feat}" for feat, task in features_wide.columns]
features_wide = features_wide.reset_index()

print(f"Wide features shape: {features_wide.shape}")
features_wide.head()

Wide features shape: (130, 443)


,participant_id,Crown_correct_Average_duration_of_Visit,Hat_correct_Average_duration_of_Visit,Iguana_correct_Average_duration_of_Visit,Rabbit_correct_Average_duration_of_Visit,Shoe_correct_Average_duration_of_Visit,T1_Prisoner-15sec_correct_Average_duration_of_Visit,T2_Ring-15sec_correct_Average_duration_of_Visit,T3_Umbrella-15 sec_correct_Average_duration_of_Visit,T4_Pen-15sec_correct_Average_duration_of_Visit,T5_Fish-15sec_correct_Average_duration_of_Visit,T6_Heart-15sec_correct_Average_duration_of_Visit,Toothbrush_correct_Average_duration_of_Visit,findDice_correct_Average_duration_of_Visit,Crown_correct_Average_duration_of_fixations,Hat_correct_Average_duration_of_fixations,Iguana_correct_Average_duration_of_fixations,Rabbit_correct_Average_duration_of_fixations,Shoe_correct_Average_duration_of_fixations,T1_Prisoner-15sec_correct_Average_duration_of_fixations,T2_Ring-15sec_correct_Average_duration_of_fixations,T3_Umbrella-15 sec_correct_Average_duration_of_fixations,T4_Pen-15sec_correct_Average_duration_of_fixations,T5_Fish-15sec_correct_Average_duration_of_fixations,T6_Heart-15sec_correct_Average_duration_of_fixations,...,Hat_total_frames,Iguana_total_frames,Rabbit_total_frames,Shoe_total_frames,T1_Prisoner-15sec_total_frames,T2_Ring-15sec_total_frames,T3_Umbrella-15 sec_total_frames,T4_Pen-15sec_total_frames,T5_Fish-15sec_total_frames,T6_Heart-15sec_total_frames,Toothbrush_total_frames,findDice_total_frames,Crown_valid_frames,Hat_valid_frames,Iguana_valid_frames,Rabbit_valid_frames,Shoe_valid_frames,T1_Prisoner-15sec_valid_frames,T2_Ring-15sec_valid_frames,T3_Umbrella-15 sec_valid_frames,T4_Pen-15sec_valid_frames,T5_Fish-15sec_valid_frames,T6_Heart-15sec_valid_frames,Toothbrush_valid_frames,findDice_valid_frames
0,vt1,467.0,1074.0,483.0,1270.0,2448.0,171.0,2533.0,342.0,NaN,417.0,NaN,NaN,2543.0,467.0,477.0,483.0,838.0,805.0,171.0,625.0,342.0,NaN,308.0,NaN,...,2618.0,3603.0,814.0,3178.0,483.0,957.0,1812.0,1836.0,1880.0,1869.0,2542.0,531.0,1926.0,2558.0,3596.0,727.0,3063.0,446.0,846.0,1803.0,1777.0,1796.0,1800.0,2452.0,436.0
1,vt10,808.0,352.0,397.0,1210.0,410.0,539.0,493.0,967.0,1844.0,1300.0,NaN,NaN,332.0,808.0,278.0,397.0,286.0,268.0,207.0,493.0,311.0,345.0,306.0,NaN,...,838.0,979.0,509.0,566.0,2496.0,2280.0,1521.0,1050.0,982.0,2581.0,935.0,493.0,558.0,608.0,706.0,368.0,390.0,1800.0,1633.0,1048.0,729.0,714.0,1803.0,683.0,369.0
2,vt100,792.0,944.0,396.0,1114.0,475.0,181.0,NaN,798.0,957.0,NaN,NaN,552.0,791.0,388.0,464.0,229.0,352.0,475.0,140.0,NaN,255.0,227.0,NaN,NaN,...,934.0,829.0,426.0,762.0,1802.0,1800.0,1330.0,447.0,1110.0,1802.0,1004.0,252.0,316.0,932.0,819.0,424.0,758.0,1801.0,1792.0,1317.0,442.0,1102.0,1801.0,987.0,250.0
3,vt101,831.0,1096.0,1275.0,1300.0,380.0,406.0,NaN,NaN,507.0,1033.0,NaN,NaN,353.0,551.0,173.0,222.0,633.0,380.0,195.0,NaN,NaN,291.0,238.0,NaN,...,689.0,747.0,536.0,623.0,1808.0,1804.0,820.0,1009.0,1355.0,1802.0,2927.0,954.0,751.0,604.0,648.0,426.0,512.0,1726.0,1690.0,681.0,908.0,1211.0,1721.0,2754.0,812.0
4,vt102,NaN,1076.0,1020.0,1067.0,1032.0,587.0,167.0,NaN,1618.0,746.0,133.0,NaN,1297.0,NaN,712.0,1020.0,525.0,329.0,222.0,167.0,NaN,801.0,224.0,133.0,...,762.0,513.0,516.0,1126.0,1802.0,1800.0,1801.0,1112.0,1833.0,1802.0,1462.0,378.0,623.0,689.0,439.0,444.0,1010.0,1759.0,1753.0,1772.0,1040.0,1768.0,1781.0,1387.0,361.0


## 3. Cross-task aggregate features

The students may generate their own aggregate features based on their literature reviews and feature engineering techniques.

In [5]:
# Mean fixation duration across all tasks
fix_dur_cols = [c for c in features_wide.columns if 'correct_Average_duration_of_fixations' in c]
if fix_dur_cols:
    features_wide['mean_fixation_duration_across_tasks'] = features_wide[fix_dur_cols].mean(axis=1)

# Total correct AOI dwell ratio across tasks
dwell_cols = [c for c in features_wide.columns if 'correct_aoi_dwell_ratio' in c]
if dwell_cols:
    features_wide['total_correct_aoi_dwell_ratio'] = features_wide[dwell_cols].mean(axis=1)

# Mean response time (time to first mouse click on correct AOI) across tasks
rt_cols = [c for c in features_wide.columns if 'correct_Time_to_first_mouse_click' in c]
if rt_cols:
    features_wide['mean_time_to_first_click'] = features_wide[rt_cols].mean(axis=1)

# Mean distractor dwell ratio
distract_cols = [c for c in features_wide.columns if 'distractor_dwell_ratio' in c]
if distract_cols:
    features_wide['mean_distractor_dwell_ratio'] = features_wide[distract_cols].mean(axis=1)

print(f"Shape after aggregate features: {features_wide.shape}")

Shape after aggregate features: (130, 447)


## 4. Merge with labels

In [6]:
dataset = labels.merge(features_wide, on='participant_id', how='left')

assert len(dataset) == len(labels), "Rows dropped during merge — check participant ID alignment"
print(f"Dataset shape: {dataset.shape}")
print(f"Participants: {dataset['participant_id'].nunique()}")

Dataset shape: (130, 465)
Participants: 130


## 5. Missing value analysis and imputation

In [7]:
# Separate feature columns from metadata/label columns
meta_cols = ['participant_id', 'total_score', 'accuracy_rate',
             PERFORMANCE_LABEL_COL, 'speed_label', 'mean_response_time_ms'] + \
            [c for c in dataset.columns if c.endswith('_correct') and len(c) < 30]
meta_cols = [c for c in meta_cols if c in dataset.columns]

X_cols = [c for c in dataset.columns if c not in meta_cols]
n_participants = len(dataset)
print(f"Feature columns before dropping: {len(X_cols)}")
print(f"Participants: {n_participants}")

null_rate = dataset[X_cols].isnull().mean().sort_values(ascending=False)

# With small N, almost every column will appear "mostly missing" under a fixed threshold.
# Only drop columns that are completely empty (100% NaN) — impute everything else.
if n_participants < 20:
    drop_threshold = 1.0   # only drop fully empty columns
    print(f"Small N ({n_participants}) — dropping only 100%-empty columns; imputing the rest.")
else:
    drop_threshold = 0.50
    print(f"Using drop threshold: {drop_threshold:.0%}")

cols_to_drop = null_rate[null_rate >= drop_threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns (≥{drop_threshold:.0%} missing).")
dataset = dataset.drop(columns=cols_to_drop)
X_cols  = [c for c in X_cols if c not in cols_to_drop]

if not X_cols:
    raise ValueError(
        f"All feature columns are completely empty (N={n_participants}). "
        "Check that notebooks 01 and 02 completed successfully."
    )

print(f"Feature columns remaining: {len(X_cols)}")

# Median imputation for remaining columns
imputer = SimpleImputer(strategy='median')
dataset[X_cols] = imputer.fit_transform(dataset[X_cols])

print(f"Remaining nulls after imputation: {dataset[X_cols].isnull().sum().sum()}")


Feature columns before dropping: 446
Participants: 130
Using drop threshold: 50%
Dropping 30 columns (≥50% missing).
Feature columns remaining: 416
Remaining nulls after imputation: 0


## 6. Outlier capping (1.5×IQR)

In [8]:
def cap_outliers_iqr(df, cols, factor=1.5):
    df = df.copy()
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR
        df[col] = df[col].clip(lower, upper)
    return df

dataset = cap_outliers_iqr(dataset, X_cols)
print("Outlier capping applied.")

Outlier capping applied.


## 7. Train/test split assignment

In [9]:
N = len(dataset)
print(f"Total participants: {N}")

# Remove participants with missing performance label
dataset_labeled = dataset.dropna(subset=[PERFORMANCE_LABEL_COL]).copy()
print(f"Participants with valid labels: {len(dataset_labeled)}")

if len(dataset_labeled) < 10:
    print("WARNING: Very few participants. All data will be used for cross-validation only — no hold-out test set.")
    dataset_labeled['split'] = 'train'
else:
    train_idx, test_idx = train_test_split(
        dataset_labeled.index,
        test_size=TEST_SIZE,
        stratify=dataset_labeled[PERFORMANCE_LABEL_COL],
        random_state=RANDOM_STATE
    )
    dataset_labeled['split'] = 'train'
    dataset_labeled.loc[test_idx, 'split'] = 'test'

print("\nSplit distribution:")
print(dataset_labeled['split'].value_counts())

Total participants: 130
Participants with valid labels: 130

Split distribution:
split
train    104
test      26
Name: count, dtype: int64


## 8. Feature scaling

In [10]:
train_mask = dataset_labeled['split'] == 'train'

scaler = StandardScaler()
scaler.fit(dataset_labeled.loc[train_mask, X_cols])

# Save unscaled version
dataset_labeled[[c + '_unscaled' for c in X_cols]] = dataset_labeled[X_cols].values

# Apply scaling
dataset_labeled[X_cols] = scaler.transform(dataset_labeled[X_cols])

joblib.dump(scaler, SCALER_PKL)
print(f"Scaler saved: {SCALER_PKL}")

Scaler saved: /home/wlsoo/github/WID2003_20260806/data/processed/scaler.pkl


/tmp/ipykernel_280214/1600392263.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_labeled[[c + '_unscaled' for c in X_cols]] = dataset_labeled[X_cols].values
/tmp/ipykernel_280214/1600392263.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_labeled[[c + '_unscaled' for c in X_cols]] = dataset_labeled[X_cols].values
/tmp/ipykernel_280214/1600392263.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

## 9. Summary and save

In [11]:
print("=== Dataset Summary ===")
print(f"Shape:              {dataset_labeled.shape}")
print(f"Feature columns:    {len(X_cols)}")
print(f"Participants:       {dataset_labeled['participant_id'].nunique()}")
print(f"High performers:    {(dataset_labeled[PERFORMANCE_LABEL_COL] == 1).sum()}")
print(f"Low performers:     {(dataset_labeled[PERFORMANCE_LABEL_COL] == 0).sum()}")
print(f"Train/test:         {dataset_labeled['split'].value_counts().to_dict()}")

dataset_labeled.to_parquet(DATASET_FINAL, index=False)
print(f"\nSaved: {DATASET_FINAL}")

=== Dataset Summary ===
Shape:              (130, 852)
Feature columns:    416
Participants:       130
High performers:    77
Low performers:     53
Train/test:         {'train': 104, 'test': 26}

Saved: /home/wlsoo/github/WID2003_20260806/data/processed/dataset_final.parquet
